1️⃣ بررسی مشخصات سیستم و سخت‌افزار


In [1]:
# ============================================================================
# بخش ۱: بررسی مشخصات سیستم و سخت‌افزار
# ============================================================================
import os
import platform
import psutil
import shutil
from pathlib import Path
import torch

print("=" * 80)
print("  گزارش مشخصات سیستم و سخت‌افزار")
print("=" * 80)

# --- اطلاعات سیستم عامل ---
print("\n اطلاعات سیستم عامل:")
print(f"   • سیستم عامل: {platform.system()} {platform.release()}")
print(f"   • نسخه پایتون: {platform.python_version()}")
print(f"   • معماری: {platform.machine()}")

# --- اطلاعات CPU ---
print("\n اطلاعات پردازنده (CPU):")
print(f"   • تعداد هسته فیزیکی: {psutil.cpu_count(logical=False)}")
print(f"   • تعداد هسته منطقی: {psutil.cpu_count(logical=True)}")
print(f"   • فرکانس فعلی: {psutil.cpu_freq().current:.2f} MHz")

# --- اطلاعات RAM ---
mem = psutil.virtual_memory()
print("\n اطلاعات حافظه (RAM):")
print(f"   • کل RAM: {mem.total / (1024**3):.2f} GB")
print(f"   • RAM مصرفی: {mem.used / (1024**3):.2f} GB ({mem.percent}%)")
print(f"   • RAM آزاد: {mem.available / (1024**3):.2f} GB")

# --- اطلاعات دیسک ---
current_dir = Path.cwd()
total, used, free = shutil.disk_usage(current_dir)
print("\n فضای ذخیره‌سازی:")
print(f"   • مسیر جاری: {current_dir}")
print(f"   • کل فضا: {total / (1024**3):.2f} GB")
print(f"   • فضای آزاد: {free / (1024**3):.2f} GB")

# --- اطلاعات GPU ---
print("\n اطلاعات کارت گرافیک (GPU):")
if torch.cuda.is_available():
    print(f"   • مدل GPU: {torch.cuda.get_device_name(0)}")
    print(f"   • تعداد GPU: {torch.cuda.device_count()}")
    print(f"   • حافظه کل: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"   • وضعیت CUDA:  فعال")
else:
    print("   • وضعیت CUDA:  غیرفعال")

print("\n" + "=" * 80)
print(" گزارش سیستم تکمیل شد")
print("=" * 80)

  گزارش مشخصات سیستم و سخت‌افزار

 اطلاعات سیستم عامل:
   • سیستم عامل: Linux 6.8.0-35-generic
   • نسخه پایتون: 3.12.9
   • معماری: x86_64

 اطلاعات پردازنده (CPU):
   • تعداد هسته فیزیکی: 32
   • تعداد هسته منطقی: 32
   • فرکانس فعلی: 2450.10 MHz

 اطلاعات حافظه (RAM):
   • کل RAM: 62.79 GB
   • RAM مصرفی: 2.08 GB (3.3%)
   • RAM آزاد: 60.71 GB

 فضای ذخیره‌سازی:
   • مسیر جاری: /home/jovyan
   • کل فضا: 192.69 GB
   • فضای آزاد: 122.72 GB

 اطلاعات کارت گرافیک (GPU):
   • مدل GPU: NVIDIA GeForce RTX 3090
   • تعداد GPU: 1
   • حافظه کل: 23.56 GB
   • وضعیت CUDA:  فعال

 گزارش سیستم تکمیل شد


2️⃣ بررسی و تحلیل دیتاست SeaShips


In [2]:
# ============================================================================
# بخش ۲: تحلیل دیتاست SeaShips
# ============================================================================
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter
import cv2
import numpy as np
from PIL import Image

# تنظیم مسیرها
BASE_DIR = Path("dataset/shipdataset/SeaShips(7000)")
IMAGE_DIR = BASE_DIR / "JPEGImages"
LABEL_DIR = BASE_DIR / "Annotations"
OUTPUT_DIR = BASE_DIR / "analysis_reports"
OUTPUT_DIR.mkdir(exist_ok=True)

print("=" * 80)
print(" تحلیل دیتاست SeaShips")
print("=" * 80)

# --- شمارش فایل‌ها ---
image_count = len(list(IMAGE_DIR.glob("*.jpg"))) + len(list(IMAGE_DIR.glob("*.png")))
label_count = len(list(LABEL_DIR.glob("*.xml")))

print(f"\n تعداد تصاویر: {image_count}")
print(f" تعداد لیبل‌ها: {label_count}")

# --- تحلیل رزولوشن ---
print("\n تحلیل رزولوشن تصاویر:")
resolutions = []
for img_path in IMAGE_DIR.glob("*.jpg"):
    img = cv2.imread(str(img_path))
    if img is not None:
        resolutions.append(img.shape[:2])

if resolutions:
    heights = [h for h, w in resolutions]
    widths = [w for h, w in resolutions]
    print(f"   • عرض میانگین: {np.mean(widths):.0f} پیکسل")
    print(f"   • ارتفاع میانگین: {np.mean(heights):.0f} پیکسل")
    print(f"   • رزولوشن غالب: {int(np.mean(widths))} × {int(np.mean(heights))}")

# --- تحلیل کلاس‌ها ---
print("\n  تحلیل کلاس‌های دیتاست:")
class_counter = Counter()
for xml_path in LABEL_DIR.glob("*.xml"):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    for obj in root.findall('object'):
        class_name = obj.find('name').text.strip()
        class_counter[class_name] += 1

total_objects = sum(class_counter.values())
print(f"    تعداد کل کلاس‌ها: {len(class_counter)}")
print(f"    تعداد کل اشیاء: {total_objects}")
print("\n   توزیع کلاس‌ها:")
for i, (cls, count) in enumerate(sorted(class_counter.items(), key=lambda x: x[1], reverse=True), 1):
    percentage = (count / total_objects) * 100
    print(f"   {i}. {cls}: {count} نمونه ({percentage:.2f}%)")

# --- بررسی تعادل کلاس‌ها ---
max_count = max(class_counter.values())
min_count = min(class_counter.values())
imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
print(f"\n  نسبت عدم تعادل (بیشترین/کمترین): {imbalance_ratio:.2f}")
if imbalance_ratio > 5:
    print("     هشدار: عدم تعادل قابل توجه در دیتاست وجود دارد")
else:
    print("    وضعیت: توزیع کلاس‌ها نسبتاً متعادل است")

print("\n" + "=" * 80)
print(" تحلیل دیتاست تکمیل شد")
print("=" * 80)

 تحلیل دیتاست SeaShips

 تعداد تصاویر: 7000
 تعداد لیبل‌ها: 7000

 تحلیل رزولوشن تصاویر:
   • عرض میانگین: 1920 پیکسل
   • ارتفاع میانگین: 1080 پیکسل
   • رزولوشن غالب: 1920 × 1080

  تحلیل کلاس‌های دیتاست:
    تعداد کل کلاس‌ها: 6
    تعداد کل اشیاء: 9221

   توزیع کلاس‌ها:
   1. ore carrier: 2199 نمونه (23.85%)
   2. fishing boat: 2190 نمونه (23.75%)
   3. bulk cargo carrier: 1952 نمونه (21.17%)
   4. general cargo ship: 1505 نمونه (16.32%)
   5. container ship: 901 نمونه (9.77%)
   6. passenger ship: 474 نمونه (5.14%)

  نسبت عدم تعادل (بیشترین/کمترین): 4.64
    وضعیت: توزیع کلاس‌ها نسبتاً متعادل است

 تحلیل دیتاست تکمیل شد


3️⃣ تبدیل و آماده‌سازی دیتاست برای YOLO


In [3]:
# ============================================================================
# بخش ۳: تبدیل و آماده‌سازی دیتاست برای YOLO
# ============================================================================
import random
import shutil
import yaml

# تنظیم مسیرها
PROJECT_DIR = Path("/home/jovyan/ship_detection_project")
YOLO_DIR = PROJECT_DIR / "datasets/SeaShips_YOLO"

# ایجاد ساختار دایرکتوری
for split in ['train', 'val', 'test']:
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

print("=" * 80)
print(" تبدیل و آماده‌سازی دیتاست برای YOLO")
print("=" * 80)

# --- تعریف کلاس‌ها ---
CLASS_MAPPING = {
    'ore carrier': 0,
    'fishing boat': 1,
    'bulk cargo carrier': 2,
    'general cargo ship': 3,
    'container ship': 4,
    'passenger ship': 5
}

print("\n  کلاس‌های تعریف‌شده:")
for cls_id, cls_name in CLASS_MAPPING.items():
    print(f"   • {cls_name}: {cls_id}")

# --- تقسیم‌بندی داده‌ها ---
print("\n تقسیم‌بندی داده‌ها (۷۰٪ آموزش، ۲۰٪ اعتبارسنجی، ۱۰٪ تست):")
random.seed(42)
image_stems = [f.stem for f in IMAGE_DIR.glob("*.jpg")]
random.shuffle(image_stems)

total = len(image_stems)
train_count = int(total * 0.7)
val_count = int(total * 0.2)
test_count = total - train_count - val_count

print(f"   • آموزش: {train_count} تصویر ({train_count/total*100:.1f}%)")
print(f"   • اعتبارسنجی: {val_count} تصویر ({val_count/total*100:.1f}%)")
print(f"   • تست: {test_count} تصویر ({test_count/total*100:.1f}%)")

train_split = image_stems[:train_count]
val_split = image_stems[train_count:train_count+val_count]
test_split = image_stems[train_count+val_count:]

# --- تابع تبدیل XML به YOLO ---
def convert_xml_to_yolo(xml_path, img_width, img_height):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    yolo_lines = []
    for obj in root.findall('object'):
        class_name = obj.find('name').text.strip()
        if class_name not in CLASS_MAPPING:
            continue
        class_id = CLASS_MAPPING[class_name]
        bndbox = obj.find('bndbox')
        xmin = int(bndbox.find('xmin').text)
        ymin = int(bndbox.find('ymin').text)
        xmax = int(bndbox.find('xmax').text)
        ymax = int(bndbox.find('ymax').text)
        
        center_x = ((xmin + xmax) / 2) / img_width
        center_y = ((ymin + ymax) / 2) / img_height
        width = (xmax - xmin) / img_width
        height = (ymax - ymin) / img_height
        
        # محدود کردن مقادیر به بازه [0, 1]
        center_x = max(0.0, min(1.0, center_x))
        center_y = max(0.0, min(1.0, center_y))
        width = max(0.0, min(1.0, width))
        height = max(0.0, min(1.0, height))
        
        yolo_lines.append(f"{class_id} {center_x:.6f} {center_y:.6f} {width:.6f} {height:.6f}")
    return yolo_lines

# --- پردازش هر بخش ---
def process_split(split_name, stems):
    img_dest = YOLO_DIR / "images" / split_name
    lbl_dest = YOLO_DIR / "labels" / split_name
    converted = 0
    
    for stem in stems:
        # پیدا کردن تصویر
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png']:
            candidate = IMAGE_DIR / f"{stem}{ext}"
            if candidate.exists():
                img_path = candidate
                break
        
        if not img_path:
            continue
        
        # پیدا کردن لیبل
        xml_path = LABEL_DIR / f"{stem}.xml"
        if not xml_path.exists():
            continue
        
        # خواندن ابعاد تصویر
        with Image.open(img_path) as img:
            img_width, img_height = img.size
        
        # تبدیل و ذخیره
        yolo_annotations = convert_xml_to_yolo(xml_path, img_width, img_height)
        lbl_file = lbl_dest / f"{stem}.txt"
        with open(lbl_file, 'w') as f:
            f.write('\n'.join(yolo_annotations))
        
        # کپی تصویر
        shutil.copy2(img_path, img_dest / img_path.name)
        converted += 1
    
    return converted

print("\n در حال پردازش داده‌ها...")
train_processed = process_split('train', train_split)
val_processed = process_split('val', val_split)
test_processed = process_split('test', test_split)

print(f"    آموزش: {train_processed} تصویر پردازش شد")
print(f"    اعتبارسنجی: {val_processed} تصویر پردازش شد")
print(f"    تست: {test_processed} تصویر پردازش شد")

# --- ایجاد فایل dataset.yaml ---
dataset_yaml = {
    'path': str(YOLO_DIR.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': list(CLASS_MAPPING.keys())
}

yaml_path = PROJECT_DIR / "dataset.yaml"
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(dataset_yaml, f, allow_unicode=True)

print(f"\n فایل پیکربندی ایجاد شد: {yaml_path}")
print("\n" + "=" * 80)
print(" آماده‌سازی دیتاست تکمیل شد")
print("=" * 80)

 تبدیل و آماده‌سازی دیتاست برای YOLO

  کلاس‌های تعریف‌شده:
   • 0: ore carrier
   • 1: fishing boat
   • 2: bulk cargo carrier
   • 3: general cargo ship
   • 4: container ship
   • 5: passenger ship

 تقسیم‌بندی داده‌ها (۷۰٪ آموزش، ۲۰٪ اعتبارسنجی، ۱۰٪ تست):
   • آموزش: 4900 تصویر (70.0%)
   • اعتبارسنجی: 1400 تصویر (20.0%)
   • تست: 700 تصویر (10.0%)

 در حال پردازش داده‌ها...
    آموزش: 4900 تصویر پردازش شد
    اعتبارسنجی: 1400 تصویر پردازش شد
    تست: 700 تصویر پردازش شد

 فایل پیکربندی ایجاد شد: /home/jovyan/ship_detection_project/dataset.yaml

 آماده‌سازی دیتاست تکمیل شد


4️⃣ آموزش مدل‌های YOLO


In [4]:
# ============================================================================
# بخش ۴: آموزش مدل‌های YOLO
# ============================================================================
from ultralytics import YOLO
from datetime import datetime

# تنظیمات آموزش
OUTPUT_ROOT = "/home/jovyan/Latest assessment"
DATA_YAML = "/home/jovyan/ship_detection_project/dataset.yaml"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# لیست مدل‌ها
models_config = [
    {'name': 'YOLOv8m', 'path': '/home/jovyan/yolov8m.pt'},
    {'name': 'YOLOv10m', 'path': '/home/jovyan/yolov10m.pt'},
    {'name': 'YOLOv11m', 'path': '/home/jovyan/yolo11m.pt'},
    {'name': 'YOLO26m', 'path': '/home/jovyan/yolo26m.pt'}
]

print("=" * 80)
print(" شروع آموزش مدل‌های YOLO")
print("=" * 80)

trained_models = []

for model_config in models_config:
    model_name = model_config['name']
    model_path = model_config['path']
    
    print(f"\n{'='*80}")
    print(f" آموزش مدل: {model_name}")
    print(f"{'='*80}")
    
    # ایجاد نام پوشه با زمان
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    project_name = f"{model_name}_SeaShips_Full7K_{timestamp}"
    save_dir = os.path.join(OUTPUT_ROOT, project_name)
    
    print(f"    مسیر ذخیره‌سازی: {save_dir}")
    print(f"    فایل دیتاست: {DATA_YAML}")
    print(f"    وزن اولیه: {model_path}")
    
    # بارگذاری و آموزش مدل
    model = YOLO(model_path)
    
    results = model.train(
        data=DATA_YAML,
        epochs=70,
        imgsz=640,
        batch=16,
        device=0,
        workers=8,
        optimizer='AdamW',
        lr0=0.01,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3.0,
        warmup_momentum=0.8,
        box=7.5,
        cls=0.5,
        dfl=1.5,
        patience=50,
        seed=42,
        deterministic=True,
        amp=True,
        project=OUTPUT_ROOT,
        name=project_name,
        exist_ok=False,
        cache=True,
        verbose=True,
        save=True,
        save_period=10,
        plots=True
    )
    
    trained_models.append({
        'name': model_name,
        'path': model_path,
        'save_dir': save_dir,
        'best_weights': os.path.join(save_dir, 'weights', 'best.pt')
    })
    
    print(f"\n آموزش {model_name} با موفقیت تکمیل شد")

print("\n" + "=" * 80)
print(" آموزش تمام مدل‌ها تکمیل شد")
print("=" * 80)

 شروع آموزش مدل‌های YOLO

 آموزش مدل: YOLOv8m
    مسیر ذخیره‌سازی: /home/jovyan/Latest assessment/YOLOv8m_SeaShips_Full7K_20260226_232459
    فایل دیتاست: /home/jovyan/ship_detection_project/dataset.yaml
    وزن اولیه: /home/jovyan/yolov8m.pt
New https://pypi.org/project/ultralytics/8.4.18 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24123MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jovyan/ship_detection_project/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None

5️⃣ ارزیابی و مقایسه مدل‌ها


In [5]:
# ============================================================================
# بخش ۵: ارزیابی و مقایسه مدل‌ها
# ============================================================================
import pandas as pd
import json
import matplotlib.pyplot as plt

print("=" * 80)
print(" ارزیابی نهایی مدل‌ها روی دیتاست تست")
print("=" * 80)

evaluation_results = []

for model_info in trained_models:
    model_name = model_info['name']
    best_weights = model_info['best_weights']
    results_dir = os.path.join(model_info['save_dir'], 'test_evaluation')
    os.makedirs(results_dir, exist_ok=True)
    
    print(f"\n{'='*80}")
    print(f" ارزیابی مدل: {model_name}")
    print(f"{'='*80}")
    
    # بارگذاری مدل
    model = YOLO(best_weights)
    
    # اجرای ارزیابی
    metrics = model.val(
        data=DATA_YAML,
        split='test',
        plots=True,
        save_json=True,
        verbose=True,
        conf=0.001,
        iou=0.7,
        project=results_dir,
        name='test_results',
        exist_ok=True
    )
    
    # استخراج معیارها
    overall_metrics = {
        'model': model_name,
        'mAP50-95': float(metrics.box.map),
        'mAP50': float(metrics.box.map50),
        'Precision': float(metrics.box.mp),
        'Recall': float(metrics.box.mr),
        'F1-Score': float(2 * (metrics.box.mp * metrics.box.mr) / (metrics.box.mp + metrics.box.mr + 1e-8))
    }
    
    evaluation_results.append(overall_metrics)
    
    print(f"\n نتایج {model_name}:")
    print(f"   • mAP50-95: {overall_metrics['mAP50-95']:.4f}")
    print(f"   • mAP50: {overall_metrics['mAP50']:.4f}")
    print(f"   • Precision: {overall_metrics['Precision']:.4f}")
    print(f"   • Recall: {overall_metrics['Recall']:.4f}")
    print(f"   • F1-Score: {overall_metrics['F1-Score']:.4f}")
    
    # ذخیره نتایج
    json_path = os.path.join(results_dir, 'test_metrics.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(overall_metrics, f, indent=4, ensure_ascii=False)

# ============================================================================
# مقایسه نهایی
# ============================================================================
print("\n" + "=" * 80)
print(" مقایسه نهایی مدل‌ها")
print("=" * 80)

df = pd.DataFrame(evaluation_results)
df = df.sort_values('mAP50-95', ascending=False).reset_index(drop=True)

print("\n جدول مقایسه‌ای:")
print(df[['model', 'mAP50-95', 'mAP50', 'Precision', 'Recall', 'F1-Score']].to_string(index=False))

# پیدا کردن بهترین مدل
best_model = df.iloc[0]
print(f"\n بهترین مدل: {best_model['model']}")
print(f"   • mAP50-95: {best_model['mAP50-95']:.4f}")
print(f"   • F1-Score: {best_model['F1-Score']:.4f}")

# ذخیره جدول مقایسه
comparison_path = os.path.join(OUTPUT_ROOT, 'models_comparison.csv')
df.to_csv(comparison_path, index=False, encoding='utf-8-sig')
print(f"\n جدول مقایسه ذخیره شد: {comparison_path}")

# ============================================================================
# تولید نمودار
# ============================================================================
print("\n در حال تولید نمودارهای مقایسه‌ای...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
model_names = df['model'].tolist()

# نمودار 1: mAP50-95
axes[0, 0].barh(model_names, df['mAP50-95'], color=colors[:len(df)], edgecolor='black', linewidth=1.5)
axes[0, 0].set_xlabel('mAP50-95', fontsize=11, fontweight='bold')
axes[0, 0].set_title('مقایسه دقت مدل‌ها (mAP50-95)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlim(0, 1.0)
axes[0, 0].grid(axis='x', linestyle='--', alpha=0.7)
for i, v in enumerate(df['mAP50-95']):
    axes[0, 0].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=9, fontweight='bold')

# نمودار 2: Precision و Recall
x = range(len(model_names))
width = 0.35
axes[0, 1].bar([i - width/2 for i in x], df['Precision'], width, label='Precision', color='#2E86AB', edgecolor='black')
axes[0, 1].bar([i + width/2 for i in x], df['Recall'], width, label='Recall', color='#A23B72', edgecolor='black')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(model_names, rotation=45, ha='right')
axes[0, 1].set_ylabel('Score', fontsize=11, fontweight='bold')
axes[0, 1].set_title('مقایسه Precision و Recall', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', linestyle='--', alpha=0.7)
axes[0, 1].set_ylim(0, 1.0)

# نمودار 3: mAP50
axes[1, 0].barh(model_names, df['mAP50'], color=colors[:len(df)], edgecolor='black', linewidth=1.5)
axes[1, 0].set_xlabel('mAP50', fontsize=11, fontweight='bold')
axes[1, 0].set_title('مقایسه دقت مدل‌ها (mAP50)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlim(0, 1.0)
axes[1, 0].grid(axis='x', linestyle='--', alpha=0.7)
for i, v in enumerate(df['mAP50']):
    axes[1, 0].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=9, fontweight='bold')

# نمودار 4: F1-Score
axes[1, 1].barh(model_names, df['F1-Score'], color=colors[:len(df)], edgecolor='black', linewidth=1.5)
axes[1, 1].set_xlabel('F1-Score', fontsize=11, fontweight='bold')
axes[1, 1].set_title('مقایسه F1-Score مدل‌ها', fontsize=12, fontweight='bold')
axes[1, 1].set_xlim(0, 1.0)
axes[1, 1].grid(axis='x', linestyle='--', alpha=0.7)
for i, v in enumerate(df['F1-Score']):
    axes[1, 1].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
chart_path = os.path.join(OUTPUT_ROOT, 'models_comparison_chart.png')
plt.savefig(chart_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

print(f" نمودار مقایسه‌ای ذخیره شد: {chart_path}")

print("\n" + "=" * 80)
print(" ارزیابی و مقایسه تکمیل شد")
print("=" * 80)

 ارزیابی نهایی مدل‌ها روی دیتاست تست

 ارزیابی مدل: YOLOv8m
Ultralytics 8.4.14 🚀 Python-3.12.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24123MiB)
Model summary (fused): 93 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 733.8±760.5 MB/s, size: 164.7 KB)
val: Scanning /home/jovyan/ship_detection_project/datasets/SeaShips_YOLO/labels/test.cache... 859 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 859/859 87.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 6.4it/s 8.4s0.1s
                   all        859       1153      0.981      0.969      0.991      0.837
           ore carrier        248        267      0.971      0.974      0.992      0.831
          fishing boat        186        284      0.991      0.933      0.992      0.793
    bulk cargo carrier        231        247      0.965      0.968      0.991      0.846
    general cargo

6️⃣ جمع‌بندی و خروجی نهایی

In [8]:
# ============================================================================
# بخش ۶: جمع‌بندی نهایی
# ============================================================================
print("=" * 80)
print(" جمع‌بندی نهایی آزمایش‌ها")
print("=" * 80)

print(f"""
 تمام مراحل با موفقیت تکمیل شد!

 مسیر نتایج: {OUTPUT_ROOT}

 فایل‌های تولیدشده:
    models_comparison.csv          ← جدول مقایسه اکسل
    models_comparison_chart.png    ← نمودار مقایسه‌ای (300 DPI)
    test_metrics.json              ← نتایج ارزیابی هر مدل

 بهترین مدل: {best_model['model']}
    mAP50-95: {best_model['mAP50-95']:.4f}
    F1-Score: {best_model['F1-Score']:.4f}

 نکات مهم:
    تمام مدل‌ها با پارامترهای یکسان آموزش داده شدند
    دیتاست به نسبت ۷۰/۲۰/۱۰ تقسیم شد
    از GPU RTX 3090 برای آموزش استفاده شد
    تعداد Epoch: 70
    Batch Size: 16
    Image Size: 640×640

  تمام خروجی‌ها قابل تکرار می‌باشند
""")

print("=" * 80)

 جمع‌بندی نهایی آزمایش‌ها

 تمام مراحل با موفقیت تکمیل شد!

 مسیر نتایج: /home/jovyan/Latest assessment

 فایل‌های تولیدشده:
    models_comparison.csv          ← جدول مقایسه اکسل
    models_comparison_chart.png    ← نمودار مقایسه‌ای (300 DPI)
    test_metrics.json              ← نتایج ارزیابی هر مدل

 بهترین مدل: YOLOv8m
    mAP50-95: 0.8367
    F1-Score: 0.9747

 نکات مهم:
    تمام مدل‌ها با پارامترهای یکسان آموزش داده شدند
    دیتاست به نسبت ۷۰/۲۰/۱۰ تقسیم شد
    از GPU RTX 3090 برای آموزش استفاده شد
    تعداد Epoch: 70
    Batch Size: 16
    Image Size: 640×640

  تمام خروجی‌ها قابل تکرار می‌باشند

